# DSCI 552 — Homework 3

**Name:** Yuanda SHI  
**USC ID:** 6635-4638-05

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from IPython.display import display

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
DATA_BASE = PROJECT_ROOT / "data" / "AReM"

print("Project directory:", PROJECT_ROOT)
print("Data directory:", DATA_BASE)
print("Data directory exists:", DATA_BASE.exists())

Project directory: /Users/sydsam/Desktop/DSCI552/HW3
Data directory: /Users/sydsam/Desktop/DSCI552/HW3/data/AReM
Data directory exists: True


## 1. Time Series Classification Part 1: Feature Creation/Extraction
### 1(a) AReM Data Set

The AReM dataset contains seven activity folders. Each instance contains six time series:

1. `avg_rss12`
2. `var_rss12`
3. `avg_rss13`
4. `var_rss13`
5. `avg_rss23`
6. `var_rss23`

Each time series contains 480 consecutive observations.

In [2]:
ACTIVITIES = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

candidate_directories = [DATA_BASE]

candidate_directories.extend(
    path for path in DATA_BASE.rglob("*")
    if path.is_dir()
)

matching_roots = [
    path
    for path in candidate_directories
    if all((path / activity).is_dir()
           for activity in ACTIVITIES)
]

assert len(matching_roots) > 0, (
    "Could not find the seven AReM activity folders "
    "under data/AReM."
)

DATA_ROOT = matching_roots[0]

print("AReM root:", DATA_ROOT)

for activity in ACTIVITIES:
    files = list((DATA_ROOT / activity).glob("dataset*.csv"))
    print(f"{activity}: {len(files)} files")

AReM root: /Users/sydsam/Desktop/DSCI552/HW3/data/AReM
bending1: 7 files
bending2: 6 files
cycling: 15 files
lying: 15 files
sitting: 15 files
standing: 15 files
walking: 15 files


In [3]:
RAW_COLUMNS = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]


def get_dataset_id(file_path):
    """
    Extract dataset number from filenames such as:
    dataset1.csv, dataset2.csv, ...
    """
    match = re.search(
        r"dataset(\d+)",
        file_path.stem,
        re.IGNORECASE
    )

    if match is None:
        raise ValueError(
            f"Cannot determine dataset number from: {file_path}"
        )

    return int(match.group(1))


def read_arem_csv(file_path, activity, dataset_id):
    """
    Read and lightly clean one AReM data file.

    Cleaning steps:
    1. Ignore comments and blank lines.
    2. Handle comma-separated files.
    3. Handle the whitespace-formatted bending2/dataset4.csv.
    4. Remove accidental trailing commas.
    5. Convert all columns to numeric.
    6. If exactly one 250-ms timestamp is missing from a
       479-row instance, reconstruct it using linear interpolation.
    7. Require exactly 480 observations after cleaning.
    """

    # ------------------------------------------------------------
    # Special case:
    # bending2/dataset4.csv uses whitespace rather than commas
    # ------------------------------------------------------------
    if activity == "bending2" and dataset_id == 4:

        frame = pd.read_csv(
            file_path,
            comment="#",
            header=None,
            names=RAW_COLUMNS,
            sep=r"\s+",
            engine="python"
        )

    # ------------------------------------------------------------
    # Standard comma-separated AReM files
    # ------------------------------------------------------------
    else:

        cleaned_rows = []

        with open(
            file_path,
            "r",
            encoding="utf-8",
            errors="replace"
        ) as file:

            for line_number, line in enumerate(
                file,
                start=1
            ):
                line = line.strip()

                # Ignore comments and blank lines
                if (
                    not line
                    or line.startswith("#")
                ):
                    continue

                # Remove accidental trailing commas
                line = line.rstrip(",")

                values = [
                    value.strip()
                    for value in line.split(",")
                ]

                if len(values) != 7:
                    raise ValueError(
                        f"\nFORMAT ERROR\n"
                        f"Activity: {activity}\n"
                        f"Dataset: {dataset_id}\n"
                        f"File: {file_path}\n"
                        f"Line: {line_number}\n"
                        f"Fields found: {len(values)}\n"
                        f"Expected: 7\n"
                        f"Content: {line}"
                    )

                cleaned_rows.append(values)

        frame = pd.DataFrame(
            cleaned_rows,
            columns=RAW_COLUMNS
        )

    # ------------------------------------------------------------
    # Convert all fields to numeric
    # ------------------------------------------------------------
    for column in RAW_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce"
        )

    # Check for missing / non-numeric values
    if frame.isna().any().any():

        bad_rows = frame[
            frame.isna().any(axis=1)
        ]

        raise ValueError(
            f"\nNON-NUMERIC OR MISSING VALUES\n"
            f"Activity: {activity}\n"
            f"Dataset: {dataset_id}\n"
            f"File: {file_path}\n\n"
            f"{bad_rows}"
        )

    # ------------------------------------------------------------
    # Sort by timestamp
    # ------------------------------------------------------------
    frame = (
        frame
        .sort_values("time")
        .reset_index(drop=True)
    )

    # ------------------------------------------------------------
    # Repair exactly one missing 250-ms observation
    # ------------------------------------------------------------
    if len(frame) == 479:

        observed_times = frame[
            "time"
        ].to_numpy()

        start_time = observed_times.min()

        expected_times = np.arange(
            start_time,
            start_time + 480 * 250,
            250
        )

        missing_times = np.setdiff1d(
            expected_times,
            observed_times
        )

        if len(missing_times) == 1:

            missing_time = missing_times[0]

            print(
                f"Cleaning {activity}/dataset{dataset_id}: "
                f"interpolating missing timestamp "
                f"{missing_time}."
            )

            missing_row = pd.DataFrame({
                "time": [missing_time],
                "avg_rss12": [np.nan],
                "var_rss12": [np.nan],
                "avg_rss13": [np.nan],
                "var_rss13": [np.nan],
                "avg_rss23": [np.nan],
                "var_rss23": [np.nan]
            })

            frame = pd.concat(
                [
                    frame,
                    missing_row
                ],
                ignore_index=True
            )

            frame = (
                frame
                .sort_values("time")
                .reset_index(drop=True)
            )

            signal_columns = RAW_COLUMNS[1:]

            frame[signal_columns] = (
                frame[signal_columns]
                .interpolate(
                    method="linear",
                    limit_direction="both"
                )
            )

        else:

            raise ValueError(
                f"\nUNEXPECTED 479-ROW FILE\n"
                f"Activity: {activity}\n"
                f"Dataset: {dataset_id}\n"
                f"File: {file_path}\n"
                f"Missing timestamps found: "
                f"{missing_times}"
            )

    # ------------------------------------------------------------
    # Final validation for each instance
    # ------------------------------------------------------------
    if len(frame) != 480:

        raise ValueError(
            f"\nFILE LENGTH ERROR\n"
            f"Activity: {activity}\n"
            f"Dataset: {dataset_id}\n"
            f"File: {file_path}\n"
            f"Rows after cleaning: {len(frame)}\n"
            f"Expected: 480"
        )

    # Verify timestamps are evenly spaced by 250 ms
    time_differences = (
        frame["time"]
        .diff()
        .dropna()
        .to_numpy()
    )

    if not np.all(
        time_differences == 250
    ):
        raise ValueError(
            f"\nTIMESTAMP ERROR\n"
            f"Activity: {activity}\n"
            f"Dataset: {dataset_id}\n"
            f"File: {file_path}\n"
            f"Not all time intervals are 250 ms."
        )

    return frame


# ================================================================
# Read all AReM instances
# ================================================================

raw_instances = {}
validation_records = []

for activity in ACTIVITIES:

    activity_files = sorted(
        (DATA_ROOT / activity)
        .glob("dataset*.csv"),
        key=get_dataset_id
    )

    for file_path in activity_files:

        dataset_id = get_dataset_id(
            file_path
        )

        frame = read_arem_csv(
            file_path,
            activity,
            dataset_id
        )

        key = (
            activity,
            dataset_id
        )

        raw_instances[key] = frame

        validation_records.append({
            "activity": activity,
            "dataset_id": dataset_id,
            "rows": frame.shape[0],
            "columns": frame.shape[1],
            "file": file_path.name
        })


# ================================================================
# Validation summary
# ================================================================

validation_df = pd.DataFrame(
    validation_records
)

print(
    "\nNumber of instances:",
    len(raw_instances)
)

print(
    "All instances have 480 rows:",
    (validation_df["rows"] == 480).all()
)

print(
    "All instances have 7 raw columns:",
    (validation_df["columns"] == 7).all()
)

print(
    "Total number of time-step records:",
    validation_df["rows"].sum()
)

print(
    "Expected total number of records:",
    88 * 480
)

display(
    validation_df.groupby("activity")
    .agg(
        instances=("dataset_id", "count"),
        min_rows=("rows", "min"),
        max_rows=("rows", "max")
    )
)

Cleaning sitting/dataset8: interpolating missing timestamp 13500.

Number of instances: 88
All instances have 480 rows: True
All instances have 7 raw columns: True
Total number of time-step records: 42240
Expected total number of records: 42240


,instances,min_rows,max_rows
activity,,,
bending1,7,480,480
bending2,6,480,480
cycling,15,480,480
lying,15,480,480
sitting,15,480,480
standing,15,480,480
walking,15,480,480


### 1(b) Train/Test Split

Use:

- datasets 1 and 2 in `bending1` and `bending2` as test data;
- datasets 1, 2, and 3 in all other activity folders as test data;
- all remaining instances as training data.

In [4]:
split_records = []

for (activity, dataset_id), frame in raw_instances.items():
    
    if activity in ["bending1", "bending2"]:
        is_test = dataset_id in [1, 2]
    else:
        is_test = dataset_id in [1, 2, 3]
    
    split_records.append({
        "activity": activity,
        "dataset_id": dataset_id,
        "instance": f"{activity}_dataset{dataset_id}",
        "split": "test" if is_test else "train"
    })


split_df = (
    pd.DataFrame(split_records)
    .sort_values(
        ["activity", "dataset_id"]
    )
    .reset_index(drop=True)
)

display(split_df)

print("\nSplit counts:")
print(split_df["split"].value_counts())

print("\nCounts by activity:")
display(
    pd.crosstab(
        split_df["activity"],
        split_df["split"]
    )
)

,activity,dataset_id,instance,split
0,bending1,1,bending1_dataset1,test
1,bending1,2,bending1_dataset2,test
2,bending1,3,bending1_dataset3,train
3,bending1,4,bending1_dataset4,train
4,bending1,5,bending1_dataset5,train
...,...,...,...,...
83,walking,11,walking_dataset11,train
84,walking,12,walking_dataset12,train
85,walking,13,walking_dataset13,train
86,walking,14,walking_dataset14,train



Split counts:
split
train    69
test     19
Name: count, dtype: int64

Counts by activity:


split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


**Train/Test Split**

The train-test split was created strictly according to the homework instructions rather than by random sampling.

The dataset identifiers within each activity folder determine whether an instance belongs to the training or test set. This preserves the exact experimental design specified in the assignment.

### 1(c) Feature Extraction

#### 1(c)i Common Time-Domain Features

Commonly used time-domain features for time-series classification include:

**Measures of location**
- Minimum
- Maximum
- Mean
- Median
- Quantiles / quartiles

**Measures of dispersion**
- Variance
- Standard deviation
- Range
- Interquartile range (IQR)
- Median absolute deviation

**Measures of distribution shape**
- Skewness
- Kurtosis

**Signal magnitude and energy**
- Root mean square (RMS)
- Signal energy
- Mean absolute value

**Temporal and change-related features**
- Mean absolute change
- Number of zero or mean crossings
- Number of peaks
- Trend/slope
- Autocorrelation

For this homework, the required features are:

- minimum
- maximum
- mean
- median
- standard deviation
- first quartile
- third quartile

#### 1(c)ii Extract the Required Time-Domain Features

For each of the six time series in every instance, extract:

1. minimum
2. maximum
3. mean
4. median
5. standard deviation
6. first quartile
7. third quartile

Therefore, each instance is represented by:

\[
6 \times 7 = 42
\]

time-domain features.

The features are used directly without normalization or standardization in this section.

In [5]:
TIME_SERIES_COLUMNS = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]


def extract_time_domain_features(frame):
    
    result = {}
    
    for series_number, column in enumerate(
        TIME_SERIES_COLUMNS,
        start=1
    ):
        series = frame[column]
        
        result[f"min{series_number}"] = (
            series.min()
        )
        
        result[f"max{series_number}"] = (
            series.max()
        )
        
        result[f"mean{series_number}"] = (
            series.mean()
        )
        
        result[f"median{series_number}"] = (
            series.median()
        )
        
        result[f"std{series_number}"] = (
            series.std(ddof=1)
        )
        
        result[f"1st_quart{series_number}"] = (
            series.quantile(0.25)
        )
        
        result[f"3rd_quart{series_number}"] = (
            series.quantile(0.75)
        )
    
    return result


feature_rows = []

for (activity, dataset_id), frame in raw_instances.items():
    
    split = split_df.loc[
        (split_df["activity"] == activity) &
        (split_df["dataset_id"] == dataset_id),
        "split"
    ].iloc[0]
    
    row = {
        "instance": f"{activity}_dataset{dataset_id}",
        "activity": activity,
        "dataset_id": dataset_id,
        "split": split
    }
    
    row.update(
        extract_time_domain_features(frame)
    )
    
    feature_rows.append(row)


feature_df = (
    pd.DataFrame(feature_rows)
    .sort_values(
        ["activity", "dataset_id"]
    )
    .reset_index(drop=True)
)

feature_df.head()

,instance,activity,dataset_id,split,min1,max1,mean1,median1,std1,1st_quart1,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,bending1_dataset1,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,bending1_dataset2,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,bending1_dataset3,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,bending1_dataset4,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,bending1_dataset5,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


In [6]:
feature_columns = [
    column
    for column in feature_df.columns
    if column not in [
        "instance",
        "activity",
        "dataset_id",
        "split"
    ]
]

print(
    "Number of instances:",
    feature_df.shape[0]
)

print(
    "Number of extracted features:",
    len(feature_columns)
)

print(
    "Missing feature values:",
    feature_df[feature_columns]
    .isna()
    .sum()
    .sum()
)

display(
    feature_df[
        ["instance"] + feature_columns
    ].head()
)

Number of instances: 88
Number of extracted features: 42
Missing feature values: 0


,instance,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,bending1_dataset1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,bending1_dataset2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,bending1_dataset3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,bending1_dataset4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,bending1_dataset5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


In [7]:
train_features = (
    feature_df[
        feature_df["split"] == "train"
    ]
    .reset_index(drop=True)
)

test_features = (
    feature_df[
        feature_df["split"] == "test"
    ]
    .reset_index(drop=True)
)

print(
    "Training instances:",
    len(train_features)
)

print(
    "Test instances:",
    len(test_features)
)

print(
    "Total:",
    len(train_features) + len(test_features)
)

Training instances: 69
Test instances: 19
Total: 88


**Feature Dataset**

Each of the 88 original time-series instances has been converted into a fixed-length vector containing 42 time-domain features.

The original temporal observations are therefore summarized into a tabular feature dataset suitable for subsequent classification tasks.

The training and test assignments were retained as metadata so that the same split can be reused in Homework 4.

#### 1(c)iii Standard Deviations and 90% Bootstrap Confidence Intervals

For each of the 42 extracted features, estimate its standard deviation across the 88 instances.

A nonparametric bootstrap is then used to construct a 90% confidence interval for each standard deviation.

The procedure is:

1. Resample the 88 instances with replacement.
2. Calculate the standard deviation of each feature in the bootstrap sample.
3. Repeat the process 2,000 times.
4. Use the 5th and 95th percentiles of the bootstrap distribution as the 90% confidence interval.

In [8]:
X_features = (
    feature_df[feature_columns]
    .to_numpy(dtype=float)
)

n_instances = X_features.shape[0]

# Point estimates of standard deviations
std_estimates = np.std(
    X_features,
    axis=0,
    ddof=1
)

# Bootstrap
N_BOOTSTRAPS = 2000

rng = np.random.default_rng(
    RANDOM_STATE
)

bootstrap_indices = rng.integers(
    low=0,
    high=n_instances,
    size=(N_BOOTSTRAPS, n_instances)
)

bootstrap_samples = (
    X_features[bootstrap_indices, :]
)

bootstrap_std = np.std(
    bootstrap_samples,
    axis=1,
    ddof=1
)

ci_lower = np.quantile(
    bootstrap_std,
    0.05,
    axis=0
)

ci_upper = np.quantile(
    bootstrap_std,
    0.95,
    axis=0
)

bootstrap_results = pd.DataFrame({
    "Feature": feature_columns,
    "Std Estimate": std_estimates,
    "90% CI Lower": ci_lower,
    "90% CI Upper": ci_upper
})

display(
    bootstrap_results.round(4)
)

,Feature,Std Estimate,90% CI Lower,90% CI Upper
0,min1,9.5700,8.2141,10.7750
1,max1,4.3944,3.3341,5.3163
2,mean1,5.3357,4.7090,5.8905
3,median1,5.4401,4.7684,6.0135
4,std1,1.7722,1.5623,1.9477
5,1st_quart1,6.1536,5.5572,6.6598
6,3rd_quart1,5.1389,4.3303,5.8597
7,min2,0.0000,0.0000,0.0000
8,max2,5.0627,4.6192,5.4059
9,mean2,1.5742,1.4005,1.7044


In [9]:
bootstrap_results_display = (
    bootstrap_results
    .set_index("Feature")
    .round(4)
)

display(
    bootstrap_results_display
)

,Std Estimate,90% CI Lower,90% CI Upper
Feature,,,
min1,9.5700,8.2141,10.7750
max1,4.3944,3.3341,5.3163
mean1,5.3357,4.7090,5.8905
median1,5.4401,4.7684,6.0135
std1,1.7722,1.5623,1.9477
1st_quart1,6.1536,5.5572,6.6598
3rd_quart1,5.1389,4.3303,5.8597
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6192,5.4059


### 1(c)iii Bootstrap Results — Interpretation

The estimated standard deviations and their 90% bootstrap confidence intervals show substantial differences in variability across the extracted time-domain features.

Several observations can be made from the results:

1. **Features from the first time series show relatively large variability.**

   In particular, `min1` has one of the largest estimated standard deviations:

   \[
   \text{SD}(\text{min1}) = 9.5700
   \]

   with a 90% bootstrap confidence interval of approximately

   \[
   [8.2141,\ 10.7750].
   \]

   Other features from the first time series also show relatively large variability, including:

   - `mean1`: SD = 5.3357, 90% CI = [4.7090, 5.8905]
   - `median1`: SD = 5.4401, 90% CI = [4.7684, 6.0135]
   - `1st_quart1`: SD = 6.1536, 90% CI = [5.5572, 6.6598]
   - `3rd_quart1`: SD = 5.1389, 90% CI = [4.3303, 5.8597]

2. **Features from the fifth time series are also highly variable.**

   For example:

   - `min5`: SD = 6.1240, 90% CI = [4.3766, 7.5031]
   - `max5`: SD = 5.7412, 90% CI = [4.7150, 6.5583]
   - `mean5`: SD = 5.6756, 90% CI = [4.4337, 6.7331]
   - `median5`: SD = 5.8138, 90% CI = [4.5243, 6.9024]
   - `1st_quart5`: SD = 6.0965, 90% CI = [4.8186, 7.1886]
   - `3rd_quart5`: SD = 5.5317, 90% CI = [4.3528, 6.5144]

   These relatively large standard deviations indicate substantial differences among the activity instances for this signal.

3. **Some minimum features show essentially no variability.**

   Both `min2` and `min4` have:

   \[
   \text{SD} = 0
   \]

   with 90% bootstrap confidence intervals of:

   \[
   [0,0].
   \]

   This means that the minimum values of time series 2 and time series 4 are identical across all 88 instances.

   The feature `min6` also shows very little variability:

   \[
   \text{SD}(\text{min6}) = 0.0458
   \]

   with a 90% bootstrap confidence interval of approximately

   \[
   [0.0000,\ 0.0785].
   \]

4. **Features from time series 2, 4, and 6 generally show lower variability than many features from time series 1, 3, and 5.**

   For example:

   - `std2`: SD = 0.8841, 90% CI = [0.8033, 0.9425]
   - `std4`: SD = 0.4582, 90% CI = [0.4203, 0.4855]
   - `std6`: SD = 0.5176, 90% CI = [0.4787, 0.5445]

   These features vary less across the 88 activity instances than many of the location-based features from time series 1 and 5.

5. **Bootstrap confidence interval widths differ substantially across features.**

   Features with larger estimated variability, such as `min1` and several features from time series 5, tend to have wider confidence intervals. This indicates greater uncertainty in estimating their population standard deviations.

   In contrast, features with very little variation, such as `min2` and `min4`, have extremely narrow or zero-width confidence intervals.

### Conclusion

The bootstrap analysis shows that the 42 extracted time-domain features differ considerably in how much they vary across the 88 activity instances.

Features from time series 1 and 5 generally show relatively high variability, while several features from time series 2, 4, and 6 are more stable across instances.

However, a large standard deviation does not necessarily imply that a feature is more useful for classification. A useful classification feature should distinguish between activity classes, not merely vary across observations.

#### 1(c)iv Selection of Three Important Time-Domain Features

Based on their interpretability and their ability to summarize different aspects of each time series, I select:

1. **Minimum**
2. **Mean**
3. **Maximum**

These three features represent the lower extreme, central tendency, and upper extreme of a time series, respectively.

In [10]:
SELECTED_FEATURE_TYPES = [
    "min",
    "mean",
    "max"
]

selected_feature_columns = []

for series_number in range(1, 7):
    
    for feature_type in SELECTED_FEATURE_TYPES:
        
        selected_feature_columns.append(
            f"{feature_type}{series_number}"
        )

print(
    "Selected feature columns:"
)

print(
    selected_feature_columns
)

selected_features_df = feature_df[
    [
        "instance",
        "activity",
        "split"
    ] + selected_feature_columns
].copy()

display(
    selected_features_df.head()
)

Selected feature columns:
['min1', 'mean1', 'max1', 'min2', 'mean2', 'max2', 'min3', 'mean3', 'max3', 'min4', 'mean4', 'max4', 'min5', 'mean5', 'max5', 'min6', 'mean6', 'max6']


,instance,activity,split,min1,mean1,max1,min2,mean2,max2,min3,...,max3,min4,mean4,max4,min5,mean5,max5,min6,mean6,max6
0,bending1_dataset1,bending1,test,37.25,40.624792,45.00,0.0,0.358604,1.30,4.0,...,29.50,0.0,0.832542,7.23,27.25,34.311292,38.25,0.0,0.570583,1.92
1,bending1_dataset2,bending1,test,38.00,42.812812,45.67,0.0,0.372437,1.22,2.0,...,29.50,0.0,0.876438,5.76,27.67,33.024583,38.50,0.0,0.571083,3.11
2,bending1_dataset3,bending1,train,35.00,43.954500,47.40,0.0,0.426250,1.70,6.5,...,29.75,0.0,0.497312,4.44,29.00,35.588458,38.50,0.0,0.493292,1.79
3,bending1_dataset4,bending1,train,33.00,42.179812,47.75,0.0,0.696042,3.00,8.5,...,30.00,0.0,0.989917,5.15,20.00,33.493917,38.67,0.0,0.613521,2.18
4,bending1_dataset5,bending1,train,33.00,41.678063,45.75,0.0,0.535979,2.83,3.0,...,28.25,0.0,0.841875,6.42,23.67,29.857083,37.50,0.0,0.383292,1.79


**Feature Selection Rationale**

I selected minimum, mean, and maximum as the three most important types of time-domain features.

- **Minimum** captures the lower extreme of the signal.
- **Maximum** captures the upper extreme of the signal.
- **Mean** captures the average or central level of the signal.

Together, these features provide a compact representation of the location and range of each time series while remaining straightforward to interpret.

The same selected feature types can later be used in the classification experiments in Homework 4.

## 2. ISLR 3.7.4

We compare a simple linear regression model with a cubic regression model:

\[
Y = \beta_0 + \beta_1X + \beta_2X^2 + \beta_3X^3 + \epsilon.
\]

An important point is that the linear model is a special case of the cubic model obtained by setting

\[
\beta_2=\beta_3=0.
\]

Therefore, the cubic model is more flexible and contains the linear model as a nested model.

### (a) True relationship is linear: Training RSS

**Answer: The cubic regression will have training RSS less than or equal to that of the linear regression.**

Because the cubic model contains all of the terms in the linear model plus the additional terms \(X^2\) and \(X^3\), it has more flexibility when fitting the training data.

If the additional terms are not useful, the cubic model can effectively set their coefficients close to zero and reproduce the linear fit. Otherwise, it may use them to fit some of the random noise in the training data.

Therefore,

\[
RSS_{\text{train,cubic}}
\leq
RSS_{\text{train,linear}}.
\]

Even though the true relationship is linear, the cubic model may achieve a slightly smaller training RSS because of its greater flexibility.

---

### (b) True relationship is linear: Test RSS

**Answer: We would generally expect the linear regression to have lower test RSS.**

Since the true relationship is linear, the linear regression model has the correct functional form.

The cubic model introduces unnecessary \(X^2\) and \(X^3\) terms. These additional degrees of freedom can fit random noise in the training data, increasing the variance of the fitted model without reducing bias in a meaningful way.

Therefore, the linear model generally has a better bias-variance trade-off and we would expect

\[
RSS_{\text{test,linear}}
<
RSS_{\text{test,cubic}}
\]

on average.

However, for a particular random training/test sample, the cubic model could occasionally obtain a lower test RSS.

---

### (c) True relationship is nonlinear: Training RSS

**Answer: The cubic regression will still have training RSS less than or equal to that of the linear regression.**

This conclusion does not depend on whether the true relationship is linear or nonlinear.

Since the cubic model contains the linear model as a special case and has additional flexibility, it can always fit the training observations at least as well as the linear model.

Thus,

\[
RSS_{\text{train,cubic}}
\leq
RSS_{\text{train,linear}}.
\]

If the true relationship contains meaningful nonlinear structure, the difference may be substantial because the cubic model may capture some of that curvature.

---

### (d) True relationship is nonlinear: Test RSS

**Answer: There is not enough information to determine which model will have the lower test RSS.**

The true relationship is known to be nonlinear, but we do not know how nonlinear it is or whether a cubic function provides a good approximation.

If the true relationship is substantially nonlinear and can be represented reasonably well by a cubic function, the cubic model may have lower bias and therefore lower test RSS.

On the other hand, if the true relationship is only slightly nonlinear, the increased variance caused by the additional polynomial terms may outweigh the reduction in bias. In that case, the simpler linear model may have lower test RSS.

Therefore, the result depends on the bias-variance trade-off and on how well the cubic model approximates the true nonlinear relationship.

---

### Summary

| Part | Expected Result |
|---|---|
| (a) Training RSS, true relationship linear | Cubic \(\leq\) Linear |
| (b) Test RSS, true relationship linear | Linear generally lower |
| (c) Training RSS, true relationship nonlinear | Cubic \(\leq\) Linear |
| (d) Test RSS, true relationship nonlinear | Cannot determine from the given information |